# 第十一章：膨脹卷積 - 多尺度上下文聚合
## Multi-Scale Context Aggregation by Dilated Convolutions
### Fisher Yu, Vladlen Koltun (ICLR 2016)

膨脹卷積（Dilated Convolution）在不增加參數、不損失解析度的情況下，指數級地擴大感受野。

**核心思想**：在卷積核元素之間插入空隙，使單層卷積覆蓋更大範圍

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# 設定隨機種子
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 11.1 一維膨脹卷積

先從一維開始理解膨脹卷積的原理。

**膨脹率 d**：卷積核元素之間的間隔
- d=1：標準卷積
- d=2：每個元素間隔 1 個位置
- d=4：每個元素間隔 3 個位置

In [ ]:
def dilated_conv1d_numpy(input_seq, kernel, dilation=1):
    """
    NumPy implementation of 1D dilated convolution
    
    Args:
        input_seq: Input sequence
        kernel: Convolution kernel
        dilation: Dilation rate
    """
    input_len = len(input_seq)
    kernel_len = len(kernel)
    
    # Effective kernel size = (k-1)*d + 1
    effective_kernel_len = (kernel_len - 1) * dilation + 1
    output_len = input_len - effective_kernel_len + 1
    
    output = []
    for i in range(output_len):
        result = 0
        for k in range(kernel_len):
            pos = i + k * dilation
            result += input_seq[pos] * kernel[k]
        output.append(result)
    
    return np.array(output)


# Test different dilation rates
signal = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=np.float32)
kernel = np.array([1, 1, 1], dtype=np.float32)  # Moving average kernel

print(f"Input signal: {signal}")
print(f"Kernel: {kernel}")
print()

for d in [1, 2, 4]:
    output = dilated_conv1d_numpy(signal, kernel, dilation=d)
    effective_size = (len(kernel) - 1) * d + 1
    print(f"Dilation d={d}: RF={effective_size}, output={output}")

In [ ]:
# Visualize 1D dilated convolution
fig, axes = plt.subplots(3, 1, figsize=(14, 9))

dilations = [1, 2, 4]
colors = ['#3498db', '#e74c3c', '#2ecc71']

for ax, d, color in zip(axes, dilations, colors):
    positions = [0, d, 2*d]
    effective_size = 2 * d + 1
    
    ax.scatter(range(10), signal, s=200, c='lightblue', edgecolors='black', zorder=2)
    for i, val in enumerate(signal):
        ax.annotate(f'{int(val)}', (i, val), textcoords="offset points", 
                   xytext=(0, 10), ha='center', fontsize=10)
    
    ax.scatter(positions, signal[positions], s=400, c=color, 
              edgecolors='black', marker='*', zorder=3, label='Kernel positions')
    
    for pos in positions:
        ax.plot([pos, pos], [0, signal[pos]], '--', color=color, alpha=0.5, linewidth=2)
    
    ax.axvspan(-0.3, positions[-1] + 0.3, alpha=0.1, color=color)
    ax.set_title(f'Dilation d={d} | Receptive Field = {effective_size}', fontsize=14)
    ax.set_xlabel('Position', fontsize=12)
    ax.set_ylabel('Value', fontsize=12)
    ax.set_xlim(-0.5, 9.5)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dilated_conv_1d.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: dilated_conv_1d.png")

## 11.2 二維膨脹卷積

在圖像處理中，我們使用二維膨脹卷積。

In [ ]:
def dilated_conv2d_numpy(input_img, kernel, dilation=1):
    """
    NumPy implementation of 2D dilated convolution
    """
    H, W = input_img.shape
    kH, kW = kernel.shape
    
    eff_kH = (kH - 1) * dilation + 1
    eff_kW = (kW - 1) * dilation + 1
    
    out_H = H - eff_kH + 1
    out_W = W - eff_kW + 1
    
    output = np.zeros((out_H, out_W))
    
    for i in range(out_H):
        for j in range(out_W):
            result = 0
            for ki in range(kH):
                for kj in range(kW):
                    img_i = i + ki * dilation
                    img_j = j + kj * dilation
                    result += input_img[img_i, img_j] * kernel[ki, kj]
            output[i, j] = result
    
    return output


# Create test image (cross pattern)
img = np.zeros((16, 16))
img[7:9, :] = 1
img[:, 7:9] = 1

# Laplacian edge detection kernel
kernel = np.array([[-1, -1, -1],
                   [-1,  8, -1],
                   [-1, -1, -1]], dtype=np.float32)

result_d1 = dilated_conv2d_numpy(img, kernel, dilation=1)
result_d2 = dilated_conv2d_numpy(img, kernel, dilation=2)
result_d3 = dilated_conv2d_numpy(img, kernel, dilation=3)

print(f"Input shape: {img.shape}")
print(f"d=1 output: {result_d1.shape}, RF: 3x3")
print(f"d=2 output: {result_d2.shape}, RF: 5x5")
print(f"d=3 output: {result_d3.shape}, RF: 7x7")

In [ ]:
# Visualize 2D dilated convolution
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

axes[0, 0].imshow(img, cmap='gray')
axes[0, 0].set_title('Input Image', fontsize=14)
axes[0, 0].axis('off')

axes[0, 1].imshow(result_d1, cmap='RdBu', vmin=-5, vmax=5)
axes[0, 1].set_title('Dilation d=1 (3x3 RF)', fontsize=14)
axes[0, 1].axis('off')

axes[1, 0].imshow(result_d2, cmap='RdBu', vmin=-5, vmax=5)
axes[1, 0].set_title('Dilation d=2 (5x5 RF)', fontsize=14)
axes[1, 0].axis('off')

axes[1, 1].imshow(result_d3, cmap='RdBu', vmin=-5, vmax=5)
axes[1, 1].set_title('Dilation d=3 (7x7 RF)', fontsize=14)
axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig('dilated_conv_2d.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: dilated_conv_2d.png")

## 11.3 感受野分析

膨脹卷積的核心優勢：**感受野指數增長**

對於核大小 k、膨脹率 d：
$$\text{RF} = (k - 1) \times d + 1$$

In [ ]:
def compute_receptive_field(kernel_size, dilation):
    """Compute single layer receptive field"""
    return (kernel_size - 1) * dilation + 1


def compute_cumulative_rf(kernel_sizes, dilations):
    """Compute cumulative receptive field for stacked layers"""
    rf = 1
    for k, d in zip(kernel_sizes, dilations):
        effective_k = (k - 1) * d + 1
        rf = rf + (effective_k - 1)
    return rf


print("=" * 60)
print("Receptive Field Analysis")
print("=" * 60)

# Standard conv stack
standard_k = [3, 3, 3, 3, 3, 3, 3]
standard_d = [1, 1, 1, 1, 1, 1, 1]
standard_rf = compute_cumulative_rf(standard_k, standard_d)

# Dilated conv (paper config)
dilated_k = [3, 3, 3, 3, 3, 3, 3]
dilated_d = [1, 1, 2, 4, 8, 16, 1]
dilated_rf = compute_cumulative_rf(dilated_k, dilated_d)

print(f"\n7-layer standard conv (d=1): RF = {standard_rf}x{standard_rf}")
print(f"7-layer dilated conv (paper): RF = {dilated_rf}x{dilated_rf}")
print(f"\nDilated conv RF is {dilated_rf/standard_rf:.1f}x larger!")

In [ ]:
# Visualize receptive field growth
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dilations_range = range(1, 17)
rf_sizes = [compute_receptive_field(3, d) for d in dilations_range]

ax1 = axes[0]
ax1.plot(dilations_range, rf_sizes, 'o-', linewidth=2, markersize=8, color='#e74c3c')
ax1.set_xlabel('Dilation Rate', fontsize=12)
ax1.set_ylabel('Receptive Field Size', fontsize=12)
ax1.set_title('Single Layer: RF vs Dilation (3x3 kernel)', fontsize=14)
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
layers = range(1, 8)
standard_rfs = [compute_cumulative_rf([3]*n, [1]*n) for n in layers]
exp_rfs = [compute_cumulative_rf([3]*n, [2**i for i in range(n)]) for n in layers]

ax2.plot(layers, standard_rfs, 'o-', label='Standard Conv (d=1)', linewidth=2, markersize=8)
ax2.plot(layers, exp_rfs, 's-', label='Dilated Conv (d=1,2,4,...)', linewidth=2, markersize=8)
ax2.set_xlabel('Number of Layers', fontsize=12)
ax2.set_ylabel('Cumulative Receptive Field', fontsize=12)
ax2.set_title('Cumulative Receptive Field Growth', fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('receptive_field_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: receptive_field_analysis.png")

## 11.4 PyTorch 實作

In [ ]:
class DilatedConvBlock(nn.Module):
    """Dilated Convolution Block"""
    
    def __init__(self, in_channels, out_channels, dilation=1):
        super(DilatedConvBlock, self).__init__()
        
        # padding = dilation keeps output size = input size
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=3,
            padding=dilation,
            dilation=dilation,
            bias=False
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))


# Test
block = DilatedConvBlock(64, 64, dilation=4)
x = torch.randn(1, 64, 32, 32)
out = block(x)

print(f"DilatedConvBlock (d=4):")
print(f"  Input:  {x.shape}")
print(f"  Output: {out.shape}")
print(f"  Params: {sum(p.numel() for p in block.parameters()):,}")

In [ ]:
class BasicContextModule(nn.Module):
    """Basic Context Module (paper design)"""
    
    def __init__(self, in_channels, out_channels):
        super(BasicContextModule, self).__init__()
        
        # Dilation config: 1, 1, 2, 4, 8, 16, 1
        dilations = [1, 1, 2, 4, 8, 16, 1]
        
        layers = []
        for i, d in enumerate(dilations):
            if i == 0:
                layers.append(DilatedConvBlock(in_channels, out_channels, d))
            else:
                layers.append(DilatedConvBlock(out_channels, out_channels, d))
        
        self.layers = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.layers(x)


# Test
context_module = BasicContextModule(64, 64)
x = torch.randn(1, 64, 64, 64)
out = context_module(x)

print(f"BasicContextModule:")
print(f"  Input:  {x.shape}")
print(f"  Output: {out.shape}")
print(f"  Cumulative RF: 67x67")
print(f"  Params: {sum(p.numel() for p in context_module.parameters()):,}")

In [ ]:
class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling"""

    def __init__(self, in_channels, out_channels, dilations=[1, 6, 12, 18]):
        super(ASPP, self).__init__()

        self.branches = nn.ModuleList()

        # 1x1 conv branch
        self.branches.append(nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        ))

        # 3x3 dilated conv branches
        for d in dilations[1:]:
            self.branches.append(nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, padding=d, dilation=d, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True)
            ))

        # Global average pooling branch
        # Note: No BatchNorm here because AdaptiveAvgPool2d(1) outputs 1x1 spatial size,
        # and BatchNorm2d requires more than 1 value per channel in training mode
        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, 1, bias=True),  # Use bias since no BN
            nn.ReLU(inplace=True)
        )

        # Fusion layer
        num_branches = len(dilations) + 1
        self.fuse = nn.Sequential(
            nn.Conv2d(out_channels * num_branches, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        size = x.shape[2:]
        features = []

        for branch in self.branches:
            features.append(branch(x))

        global_feat = self.global_pool(x)
        global_feat = F.interpolate(global_feat, size=size, mode='bilinear', align_corners=False)
        features.append(global_feat)

        x = torch.cat(features, dim=1)
        x = self.fuse(x)

        return x


# Test
aspp = ASPP(256, 256)
x = torch.randn(1, 256, 32, 32)
out = aspp(x)

print(f"ASPP Module:")
print(f"  Input:  {x.shape}")
print(f"  Output: {out.shape}")
print(f"  Dilations: [1, 6, 12, 18] + global pooling")
print(f"  Params: {sum(p.numel() for p in aspp.parameters()):,}")

## 11.5 完整分割網路

In [ ]:
class DilatedSegNet(nn.Module):
    """Segmentation Network with Dilated Convolutions"""
    
    def __init__(self, num_classes=21, in_channels=3):
        super(DilatedSegNet, self).__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        
        # Dilated layers
        self.dilated_layers = nn.Sequential(
            DilatedConvBlock(256, 256, dilation=1),
            DilatedConvBlock(256, 256, dilation=2),
            DilatedConvBlock(256, 256, dilation=4),
            DilatedConvBlock(256, 256, dilation=8),
            DilatedConvBlock(256, 256, dilation=16),
        )
        
        # Classifier
        self.classifier = nn.Conv2d(256, num_classes, 1)
    
    def forward(self, x):
        size = x.shape[2:]
        
        x = self.encoder(x)
        x = self.dilated_layers(x)
        x = self.classifier(x)
        
        x = F.interpolate(x, size=size, mode='bilinear', align_corners=False)
        
        return x


# Test
model = DilatedSegNet(num_classes=21).to(device)
x = torch.randn(1, 3, 128, 128).to(device)
out = model(x)

print(f"DilatedSegNet:")
print(f"  Input:  {x.shape}")
print(f"  Output: {out.shape}")
print(f"  Params: {sum(p.numel() for p in model.parameters()):,}")

## 11.6 比較分析

In [ ]:
# Compare architectures
def compare_architectures():
    results = []
    
    # Standard 3x3 conv stack (7 layers)
    standard = nn.Sequential(*[nn.Conv2d(64, 64, 3, padding=1) for _ in range(7)])
    rf_standard = compute_cumulative_rf([3]*7, [1]*7)
    results.append(('Standard Conv (7 layers)', sum(p.numel() for p in standard.parameters()), rf_standard))
    
    # Large kernel (15x15)
    large_kernel = nn.Conv2d(64, 64, 15, padding=7)
    results.append(('Large Kernel (15x15)', sum(p.numel() for p in large_kernel.parameters()), 15))
    
    # Dilated conv (paper config)
    dilated = BasicContextModule(64, 64)
    rf_dilated = 67
    results.append(('Dilated Conv (paper)', sum(p.numel() for p in dilated.parameters()), rf_dilated))
    
    return results

results = compare_architectures()

print("Architecture Comparison:")
print("=" * 60)
print(f"{'Method':<25} {'Parameters':>15} {'RF':>10}")
print("=" * 60)
for name, params, rf in results:
    print(f"{name:<25} {params:>15,} {rf:>10}x{rf}")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

methods = ['Standard\n(7 layers)', 'Large Kernel\n(15x15)', 'Dilated Conv\n(paper)']
params = [r[1] for r in results]
rfs = [r[2] for r in results]

ax1 = axes[0]
bars = ax1.bar(methods, params, color=['#3498db', '#e74c3c', '#2ecc71'])
ax1.set_ylabel('Parameters', fontsize=12)
ax1.set_title('Parameter Count Comparison', fontsize=14)
for bar, p in zip(bars, params):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10000,
            f'{p:,}', ha='center', fontsize=10)

ax2 = axes[1]
bars = ax2.bar(methods, rfs, color=['#3498db', '#e74c3c', '#2ecc71'])
ax2.set_ylabel('Receptive Field Size', fontsize=12)
ax2.set_title('Receptive Field Comparison', fontsize=14)
for bar, r in zip(bars, rfs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{r}x{r}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('architecture_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: architecture_comparison.png")

## 11.7 網格效應視覺化

In [ ]:
def visualize_gridding_effect():
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    grid_size = 16
    
    # d=2 first layer
    grid1 = np.zeros((grid_size, grid_size))
    for i in range(0, grid_size, 2):
        for j in range(0, grid_size, 2):
            grid1[i, j] = 1
    axes[0].imshow(grid1, cmap='Blues', vmin=0, vmax=1)
    axes[0].set_title('Layer 1: d=2\n(Sampled positions)', fontsize=12)
    axes[0].axis('off')
    
    # d=2 on top of d=2 -> gridding!
    grid2 = np.zeros((grid_size, grid_size))
    for i in range(0, grid_size, 4):
        for j in range(0, grid_size, 4):
            grid2[i, j] = 1
    axes[1].imshow(grid2, cmap='Reds', vmin=0, vmax=1)
    axes[1].set_title('Layer 2: d=2 on Layer 1\n(Gridding effect!)', fontsize=12)
    axes[1].axis('off')
    
    # HDC [1, 2, 5] - better coverage
    grid3 = np.zeros((grid_size, grid_size))
    for d in [1, 2, 5]:
        for i in range(0, grid_size, d):
            for j in range(0, grid_size, d):
                grid3[i, j] = 1
    axes[2].imshow(grid3, cmap='Greens', vmin=0, vmax=1)
    axes[2].set_title('HDC [1,2,5]\n(Better coverage)', fontsize=12)
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.savefig('gridding_effect.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: gridding_effect.png")

visualize_gridding_effect()

## 11.8 總結

In [ ]:
# Summary table
fig, ax = plt.subplots(figsize=(12, 6))

methods = ['Standard Conv', 'Pooling', 'Large Kernel', 'Dilated Conv']
rf_growth = ['Linear', 'Fast', 'Linear', 'Exponential']
resolution = ['Full', 'Reduced', 'Full', 'Full']
params_list = ['Low', 'None', 'High', 'Low']

cell_text = [[methods[i], rf_growth[i], resolution[i], params_list[i]] for i in range(len(methods))]

table = ax.table(
    cellText=cell_text,
    colLabels=['Method', 'RF Growth', 'Resolution', 'Parameters'],
    cellLoc='center',
    loc='center',
    colColours=['#3498db']*4
)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.5, 2)

for j in range(4):
    table[(4, j)].set_facecolor('#2ecc71')
    table[(4, j)].set_text_props(weight='bold')

ax.axis('off')
ax.set_title('Comparison of Methods for Expanding Receptive Field', fontsize=16, pad=20)

plt.tight_layout()
plt.savefig('method_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: method_comparison.png")

In [ ]:
print("=" * 60)
print("Chapter 11: Dilated Convolutions - Complete!")
print("=" * 60)
print("\nKey Points:")
print("1. Dilated conv inserts gaps between kernel elements")
print("2. RF = (k-1) * d + 1, where k=kernel size, d=dilation")
print("3. Stacked dilated conv achieves exponential RF growth")
print("4. No extra parameters, no resolution loss")
print("5. Used in semantic segmentation, WaveNet, TCN")
print("\nSaved figures:")
print("  1. dilated_conv_1d.png")
print("  2. dilated_conv_2d.png")
print("  3. receptive_field_analysis.png")
print("  4. architecture_comparison.png")
print("  5. gridding_effect.png")
print("  6. method_comparison.png")